In [5]:
import pandas as pd
import numpy as np
import itertools
from typing import List, Dict, Any

# ====================== ALL FUNCTIONS (same as yours) ======================
def preprocess_timeseries(
    df: pd.DataFrame,
    timestamp_col: str = "time",
    selected_streams: List[str] = None
) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()
   
    if selected_streams is None:
        selected_streams = [col for col in df.columns if col != timestamp_col]
   
    selected_streams = [s.strip() for s in selected_streams]
   
    columns_to_select = [timestamp_col] + selected_streams
    df_selected = df[columns_to_select].copy()
    
    df_selected[timestamp_col] = pd.to_datetime(
        df_selected[timestamp_col], unit='s', errors='coerce'
    )
    df_selected = df_selected.sort_values(timestamp_col)
    df_selected = df_selected.set_index(timestamp_col)
    
    numeric_cols = df_selected.select_dtypes(include=[np.number]).columns
    if not numeric_cols.empty:
        df_selected[numeric_cols] = df_selected[numeric_cols].interpolate(
            method='linear', limit_direction='both'
        )
    df_selected = df_selected.dropna(how='all')
    return df_selected


def create_rolling_windows(
    df: pd.DataFrame,
    window_size: int = 100,
    step_size: int = 50
) -> List[pd.DataFrame]:
    windows = []
    n = len(df)
    for start in range(0, n - window_size + 1, step_size):
        window_df = df.iloc[start:start + window_size].copy()
        windows.append(window_df)
    return windows


def compute_window_correlations(
    windows: List[pd.DataFrame],
    method: str = "pearson"
) -> List[Dict[str, Any]]:
    results = []
    for idx, window in enumerate(windows):
        if len(window) < 2:
            continue
        corr_matrix = window.corr(method=method)
        results.append({
            "window_index": idx,
            "start_time": window.index[0],
            "end_time": window.index[-1],
            "window_size": len(window),
            "correlation_matrix": corr_matrix
        })
    return results


def compare_correlation_changes(correlation_results: List[Dict]) -> List[Dict]:
    changes = []
    if len(correlation_results) < 2:
        return changes
    for i in range(1, len(correlation_results)):
        prev = correlation_results[i-1]
        curr = correlation_results[i]
        prev_matrix = prev["correlation_matrix"]
        curr_matrix = curr["correlation_matrix"]
        common_streams = sorted(list(set(prev_matrix.columns) & set(curr_matrix.columns)))
        for s1, s2 in itertools.combinations(common_streams, 2):
            prev_corr = prev_matrix.loc[s1, s2]
            curr_corr = curr_matrix.loc[s1, s2]
            delta = abs(curr_corr - prev_corr)
            changes.append({
                "window_index": curr["window_index"],
                "start_time": curr["start_time"],
                "end_time": curr["end_time"],
                "stream_1": s1,
                "stream_2": s2,
                "previous_corr": float(prev_corr),
                "current_corr": float(curr_corr),
                "delta": float(delta)
            })
    return changes


def generate_alerts(
    changes: List[Dict],
    strong_corr_threshold: float = 0.7,
    weak_corr_threshold: float = 0.4,
    delta_threshold: float = 0.25
) -> List[Dict]:
    alerts = []
    for change in changes:
        if change["delta"] >= delta_threshold:
            prev = change["previous_corr"]
            curr = change["current_corr"]
            if prev >= strong_corr_threshold and curr <= weak_corr_threshold:
                level = "HIGH"
                reason = f"Strong correlation dropped sharply ({prev:.3f} → {curr:.3f})"
            elif (prev > 0 and curr < 0) or (prev < 0 and curr > 0):
                level = "MEDIUM"
                reason = f"Sign change detected ({prev:.3f} → {curr:.3f})"
            else:
                level = "LOW"
                reason = f"Significant change ({prev:.3f} → {curr:.3f})"
            alert = change.copy()
            alert["alert_level"] = level
            alert["reason"] = reason
            alerts.append(alert)
    return alerts


def detect_correlation_change_alert(
    df: pd.DataFrame,
    timestamp_col: str = "time",
    selected_streams: List[str] = None,
    window_size: int = 100,
    step_size: int = 50,
    method: str = "pearson",
    strong_corr_threshold: float = 0.7,
    weak_corr_threshold: float = 0.4,
    delta_threshold: float = 0.25
) -> Dict[str, Any]:
    processed_data = preprocess_timeseries(df, timestamp_col, selected_streams)
    windows = create_rolling_windows(processed_data, window_size, step_size)
    correlation_results = compute_window_correlations(windows, method)
    changes = compare_correlation_changes(correlation_results)
    alerts = generate_alerts(changes, strong_corr_threshold, weak_corr_threshold, delta_threshold)
    
    return {
        "processed_data": processed_data,
        "windows": windows,
        "correlation_results": correlation_results,
        "changes": changes,
        "alerts": alerts,
        "summary": {
            "total_windows": len(windows),
            "total_changes": len(changes),
            "total_alerts": len(alerts),
            "window_size": window_size,
            "step_size": step_size
        }
    }


# ====================== RUN IT WITH CORRELATION MATRICES ======================
if __name__ == "__main__":
    df = pd.read_csv("simple.csv")
   
    result = detect_correlation_change_alert(
        df=df,
        timestamp_col="time",
        selected_streams=["s1", "s2", "s3"],
        window_size=100,
        step_size=50,
        delta_threshold=0.25
    )

    print("=== Correlation Change Detection Summary ===")
    print(f"Total windows : {result['summary']['total_windows']}")
    print(f"Total changes : {result['summary']['total_changes']}")
    print(f"Total alerts  : {result['summary']['total_alerts']}\n")

    # ==================== PRINT ALL CORRELATION MATRICES ====================
    print("=== Correlation Matrices for All Windows ===\n")
    
    for res in result["correlation_results"]:
        print(f"Window {res['window_index']:2d} | "
              f"From: {res['start_time']}  →  {res['end_time']} "
              f"(Size: {res['window_size']})")
        # Round to 4 decimal places for clean reading
        print(res["correlation_matrix"].round(4))
        print("-" * 70)

    # ==================== SHOW ALERTS ====================
    if result["alerts"]:
        alerts_df = pd.DataFrame(result["alerts"])
        print("\n=== Alerts Found ===")
        print(alerts_df[["window_index", "start_time", "stream_1", "stream_2",
                         "previous_corr", "current_corr", "delta", "alert_level"]].round(4))
    else:
        print("\nNo significant correlation changes detected with current thresholds.")

=== Correlation Change Detection Summary ===
Total windows : 19
Total changes : 54
Total alerts  : 14

=== Correlation Matrices for All Windows ===

Window  0 | From: 1970-01-01 00:00:00  →  1970-01-01 00:01:39 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.9534  1.0000
s2 -0.9534  1.0000 -0.9534
s3  1.0000 -0.9534  1.0000
----------------------------------------------------------------------
Window  1 | From: 1970-01-01 00:00:50  →  1970-01-01 00:02:29 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.9605  1.0000
s2 -0.9605  1.0000 -0.9605
s3  1.0000 -0.9605  1.0000
----------------------------------------------------------------------
Window  2 | From: 1970-01-01 00:01:40  →  1970-01-01 00:03:19 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.4959  1.0000
s2 -0.4959  1.0000 -0.4959
s3  1.0000 -0.4959  1.0000
----------------------------------------------------------------------
Window  3 | From: 1970-01-01 00:02:30  →  1970-01-01 00:04:09 (Size: 100)
        s1      s2